# 2025-26 Project - Foundations of Computer Science

**Project by:**
- Herandez, Nicole Winy </br>
- Ortega, Azriel Matthew </br>
- Tibayan, Meryann Joy

In [ ]:
# import libraries
import pandas as pd
import os
import json
import numpy as np

### 1. Create a single dataframe with the concatenation of all input csv files, adding a column called country

In [ ]:
# read csv files with read_csv function from pandas
videos_folder = "videos"

df_cavideos = pd.read_csv(f"{videos_folder}/CAvideos.csv")
df_devideos = pd.read_csv(f"{videos_folder}/DEvideos.csv")
df_frvideos = pd.read_csv(f"{videos_folder}/FRvideos.csv")
df_gbvideos = pd.read_csv(f"{videos_folder}/GBvideos.csv")
df_invideos = pd.read_csv(f"{videos_folder}/INvideos.csv")
df_usvideos = pd.read_csv(f"{videos_folder}/USvideos.csv")

# After experimenting, it seems that using default encoding(utf-8) and using encoding errors
# to skip some bytes that cant be read is better than finding a specific encoding for each language
# However, this must be researched more for example: If this only a computer specific problem, or 
# there is a way to read everything properly without having to use encoding errors

df_jpvideos = pd.read_csv(f"{videos_folder}/JPvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_krvideos = pd.read_csv(f"{videos_folder}/KRvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_mxvideos = pd.read_csv(f"{videos_folder}/MXvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_ruvideos = pd.read_csv(f"{videos_folder}/RUvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")

They say it is hard to determine the correct encoding for each text file, especially if it is not utf-8 </br>
Reference https://community.dataquest.io/t/how-do-we-know-the-encoding-to-use-with-pd-read-csv-chardet-doesnt-help/467614/4 </br>

You have to do trial and error with all of the encodings to check which one works best. As for my testing 11/20/2025, utf-8 is best for all with the encoding_errors set to replace.

Here is the docs for checking all encodings : https://docs.python.org/3/library/codecs.html#standard-encodings

In [ ]:
# Efficient code version (with concatenation process

# Import all csv files, read and compile them into one dataframe
# With the addition of adding a new column 'country' to determine from what csv file it came from

dataframe_lists = []
for file in os.listdir(videos_folder):
    if file[9:] == "csv":
        print(f"Reading {file}")
        df_temp = pd.read_csv(f"{videos_folder}/{file}", encoding_errors="replace")
        df_temp["country"] = file[:2] # add country column
        dataframe_lists.append(df_temp)

df_videos = pd.concat(dataframe_lists) # concatenate everything into one dataframe
df_videos = df_videos.reset_index(drop=True) # reset index

In [ ]:
df_videos

### 2. Extract all videos that have no tag.

 I extracted videos with no tags by cleaning the tags column first: I handled nulls, converted values to strings, trimmed spaces, and normalized case. Then I filtered values like [none], none, empty string. 

In [ ]:
no_tag_videos = df_videos[df_videos["tags"].fillna("").astype(str).str.strip().str.lower().isin(["[none]", "none", "", "nan", "null"])]
no_tag_videos #print all videos that have tags [none]

I also created a deduplicated version by video_id to count unique videos.

In [ ]:
no_tag_videos_unici = no_tag_videos.drop_duplicates(subset=["video_id"])
no_tag_videos_unici #there are 23992 unique videos that have tags [none]

### 3. For each channel, determine the total number of views

I computed total views per channel in a way that avoids double counting. Since the same video appears on multiple trending dates, I first took the maximum views for each (channel_title, video_id) pair, then I summed those values by channel.

In [ ]:
# We take the max views per video within each channel, then sum per channel.
total_views_per_channel = (
    df_videos.groupby(["channel_title", "video_id"])["views"].max()
    .groupby("channel_title").sum()
    .sort_values(ascending=False)
)
total_views_per_channel


### 4. Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.

I created an excluded dataframe with rows where both comments and ratings are disabled, or where the video is marked as error/removed. After saving those rows, I removed them from the main dataframe so later analyses use cleaner data.

In [ ]:
excluded = df_videos[(df_videos["comments_disabled"] & df_videos["ratings_disabled"]) | df_videos["video_error_or_removed"]]
excluded


In [ ]:
df_videos = df_videos.drop(excluded.index)
df_videos

### 5. Add a like_ratio column storing the ratio between the number of likes and of dislikes

I added like_ratio = likes / dislikes. To avoid division by zero, I replaced dislikes = 0 with NaN. This keeps undefined ratios explicit instead of creating invalid infinite values.

In [ ]:
# Replace dislikes == 0 with NaN to avoid division-by-zero when computing like_ratio (likes / dislikes)
df_videos["like_ratio"] = df_videos["likes"] / df_videos["dislikes"].replace(0, np.nan) 

df_videos[["title", "likes", "dislikes", "like_ratio"]].head(10)

In [21]:
#if needed: total of disabled ratings
df_videos[df_videos["dislikes"] == 0]["ratings_disabled"].value_counts()

In [22]:
#percentage
df_videos[df_videos["dislikes"] == 0]["ratings_disabled"].value_counts(normalize=True) * 100

In [23]:
# how many rows have like_ratio = NaN (number of videos that have number of dislikes = 0)
num_nan_ratio = df_videos["like_ratio"].isna().sum()
print("Rows with like_ratio NaN:", num_nan_ratio)


### 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

I converted publish_time to datetime in UTC, then clustered times into 10-minute bins using floor rounding.

In [ ]:
pt = pd.to_datetime(df_videos["publish_time"], utc=True, errors="coerce")
start = pt.dt.floor("10min")
df_videos["publish_interval"] = (
    start.dt.strftime("%H:%M") + " - " +
    (start + pd.Timedelta(minutes=30)).dt.strftime("%H:%M")
)

df_videos[["publish_time", "publish_interval"]]

### 7. For each interval, determine the number of videos, average number of likes and of dislikes.

I computed interval statistics: number of videos, average likes, and average dislikes. I kept two perspectives: one by unique videos per interval, and one by trending appearances (rows). The unique-video version avoids overweighting videos that appear many times; the row-based version measures exposure frequency.

In [ ]:
# Unique Videos (Counting unique videos per interval)
interval_video_level = df_videos.groupby(["publish_interval", "video_id"], as_index=False).agg(
    likes_per_video = ("likes", "mean"),
    dislikes_per_video = ("dislikes", "mean")
)

interval_stats = interval_video_level.groupby("publish_interval").agg(
    number_of_videos = ("video_id", "count"),
    average_likes = ("likes_per_video", "mean"),
    average_dislikes = ("dislikes_per_video", "mean")
)

interval_stats.sort_index().head(50)

In [ ]:
# By Rows (Counting trending apperances)
interval_stats = df_videos.groupby("publish_interval").agg(
    number_of_videos = ("video_id", "count"),
    average_likes = ("likes", "mean"),
    average_dislikes = ("dislikes", "mean")
)

interval_stats.sort_index().head(50)

### 8. For each tag, determine the number of videos. Notice that tags contains a string with several tags

Setup tag column in our dataframe first <br>
Split the string into array with '|' as the separator <br>

Explode dataframe so that each row wiill have one tag, it will duplicate rows with many tags. This is the approach because it is useful when grouping by tags.

In [ ]:
df_videos_tags = df_videos.copy()

In [ ]:
df_videos_tags["tags"] = df_videos_tags["tags"].str.split('|')

In [ ]:
df_videos_tags = df_videos_tags[["video_id", "tags", "like_ratio", "country", "trending_date"]]

In [ ]:
df_videos_tags_explode = df_videos_tags.explode('tags') 

In [ ]:
# clean dataset by removing all extra characters such as double quotations (")
df_videos_tags_explode["tags"] = (
    df_videos_tags_explode["tags"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
    .str.lower()
) 

In [ ]:
df_videos_tags_explode = df_videos_tags_explode[
    (df_videos_tags_explode["tags"].str.strip() != "") &
    (~df_videos_tags_explode["tags"].str.strip().str.lower().isin(["[none]", "none", "nan", "null"]))
]

In [ ]:
# Videos with no tags are excluded
df_videos_tags_explode.groupby(by="tags")['video_id'].nunique()

### 9. Find the tags with the largest number of videos

In [ ]:
df_videos_tags_explode[df_videos_tags_explode["tags"] != "[none]"].groupby(by="tags")['video_id'].nunique().sort_values(ascending=False).head(1)

### 10. For each (tag, country) pair, compute average ratio likes/dislikes

In [ ]:
df_videos_tags_only = df_videos_tags_explode[(df_videos_tags_explode["tags"] != "[none]")]

In [ ]:
pd.DataFrame(df_videos_tags_only.groupby(by=["tags","country"])["like_ratio"].mean())

### 11. For each (trending_date, country) pair, the video with the largest number of views

In [ ]:
# For each (trending_date, country) group, idxmax() returns the index of the row with the highest views (one row per group).
max_views = df_videos.groupby(["trending_date", "country"])["views"].idxmax()
max_views

In [ ]:
df_videos_group = df_videos.loc[max_views]
print(df_videos_group)

In [ ]:
# Sort columns, to display Country, Trending Date and then Views First.
priority_columns = ["trending_date","country", "views"]
other_columns = [col for col in df_videos.columns if col not in priority_columns]

df_videos_group[priority_columns + other_columns]

### 12. Divide trending_date into three columns: year, month, day

In [ ]:
df_videos.info()

In [ ]:
df_videos['trending_date'] = pd.to_datetime(df_videos['trending_date'], format='%y.%d.%m')

df_videos['year'] = df_videos['trending_date'].dt.year
df_videos['month'] = df_videos['trending_date'].dt.month
df_videos['day'] = df_videos['trending_date'].dt.day

print(df_videos[['trending_date', 'year', 'month', 'day']].head())

### 13. For each (month, country) pair, the video with the largest number of views

In [ ]:
max_views_indices = df_videos.groupby(['month', 'country'])['views'].idxmax()

df_max_views = df_videos.loc[max_views_indices]

df_max_views

### 14. Read all json files with the video categories

In [ ]:
json_folder = 'categories'
df_list = []

for file in os.listdir(json_folder):
    if file.endswith(".json"):
        country_code = file.split('_')[0] 
        
        with open(os.path.join(json_folder, file), 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        df_temp = pd.json_normalize(data['items'])
        df_temp['id'] = df_temp['id'].astype(int)
        df_temp['country'] = country_code
        df_list.append(df_temp)

df_categories = pd.concat(df_list, ignore_index=True)

df_categories


### 15. For each country, determine how many videos have a category that is not assignable

In [ ]:
df_merged = pd.merge(df_videos, df_categories, left_on = ['country', 'category_id'], right_on = ['country', 'id'],how='left')
df_merged.shape
df_merged

In [ ]:
# Videos where category information is missing (assignable is NaN)
missing_info = df_merged[df_merged['snippet.assignable'].isna()]
missing_info[['video_id','category_id','country','snippet.assignable']]


In [ ]:
#Missing categories in JSON files for each country
missing_categories_by_country = (
    df_merged[df_merged['snippet.assignable'].isna()]
    .groupby('country')
    ['category_id'].unique() 
    .reset_index(name='missing_category_id')
)

print(missing_categories_by_country)

In [ ]:
#video count for each country where assignable is NaN
df_Nan_assignable = (
    df_merged[df_merged['snippet.assignable'].isna()]
    .groupby('country')
    ['video_id'].nunique()
    .reset_index(name='nan_assignable_count')
    .sort_values('nan_assignable_count', ascending=False)
)

print(df_Nan_assignable)

In [ ]:
# count of unique videos that are not assignable for each country, including those with NaN assignable
df_Nan_not_assignable = (df_merged[(df_merged['snippet.assignable'] == False) | (df_merged['snippet.assignable'].isna())]
    .groupby('country')
    ['video_id'].nunique()
    .reset_index(name='Nan_non_assignable_count')
    .sort_values('Nan_non_assignable_count', ascending = False) 
)

df_Nan_not_assignable

In [ ]:
# videos that are not assignable for each country (including countries with 0)
all_countries = sorted(df_videos['country'].unique())

df_non_assignable = (df_merged[df_merged['snippet.assignable'] == False]
    .groupby('country')['video_id'].nunique()
    .reindex(all_countries, fill_value=0)
    .reset_index(name='non_assignable_count')
    .sort_values('non_assignable_count', ascending=False)
)

df_non_assignable